# Exactitud de Símbolo Braille frente a una transcripción humana

Se mide el BSA del sistema completo sobre las Variaciones Goldberg (Bach, BWV 988) del proyecto Open Goldberg. El audio es la grabación de Kimiko Ishizaka y la referencia es la edición en musicografía Braille de la misma obra, elaborada por un servicio de transcripción Braille. Las dos se publicaron bajo licencia Creative Commons Zero.

Las notas vienen del notebook `amt-goldberg-transcribe` en Kaggle, que transcribió cada pieza con el modelo original y con el modelo del ajuste fino. Aquí se traducen a Braille con el código del repositorio y se comparan con la referencia. No hace falta GPU.

La armadura y el compás de cada pieza se toman del encabezado de la edición, que es la misma información que tendría un usuario con la partitura. El tempo se estima de la propia grabación.

## 1. Código y datos

In [1]:
import json, sys, urllib.request
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from evaluation.reference import parse_reference
from evaluation.reference_eval import evaluate_piece, merge

REFERENCIA_URL = "https://opengoldbergvariations.org/Bach-Goldberg-Variations-Braille-Edition.brf"
NOTAS = ROOT / "results" / "oe5_notas"
SALIDA = ROOT / "results"

brf = urllib.request.urlopen(REFERENCIA_URL).read().decode("ascii")
piezas = parse_reference(brf)
print(len(piezas), "piezas en la referencia")

32 piezas en la referencia


## 2. Referencia

Cada pieza de la edición trae su armadura, su indicación de compás y sus compases por mano. Las repeticiones se expanden como las toca la intérprete y se quitan los signos que el sistema no escribe, que son las repeticiones, las casillas de primera y segunda vez y los ornamentos y articulaciones fuera de las 28 reglas.

Dos piezas tienen particularidades que el sistema no representa. La Variación 26 es polimétrica, con la mano derecha en 18/16 y la izquierda en 3/4. La Variación 30 empieza con anacrusa. Las dos se evalúan igual y se marcan en los resultados.

In [2]:
nombres = ["Aria"] + [f"Variación {i}" for i in range(1, 31)] + ["Aria da capo"]
filas = []
for p in piezas:
    filas.append({"pieza": p.index, "nombre": nombres[p.index], "armadura": p.fifths,
                  "compas": f"{p.beats}/{p.beat_type}", "compases": len(p.measures),
                  "anacrusa": p.has_pickup})
pd.DataFrame(filas).set_index("pieza")

,nombre,armadura,compas,compases,anacrusa
pieza,,,,,
0,Aria,1,3/4,32,False
1,Variación 1,1,3/4,32,False
2,Variación 2,1,2/4,34,False
3,Variación 3,1,12/8,16,False
4,Variación 4,1,3/8,34,False
5,Variación 5,1,3/4,32,False
6,Variación 6,1,3/8,36,False
7,Variación 7,1,6/8,32,False
8,Variación 8,1,3/4,32,False


## 3. Evaluación

Para cada pieza y modelo se decide si la grabación toca las repeticiones comparando cuántas notas se transcribieron con cuántas escribe la referencia con y sin repeticiones. Como las dos hipótesis difieren en un factor cercano a dos, la frontera es la media geométrica de ambas cuentas. El tempo se estima con los pulsos que suenan entre el primer y el último ataque. Las notas se cuantizan con la armadura y el compás de la edición y se traducen con un compás por paralela, igual que en la referencia. El BSA se calcula mano por mano con alineamiento global.

Se evalúan dos configuraciones del cuantizador. Con `legato = 0` la figura sigue lo que la nota sonó, que es el comportamiento del sistema. Con `legato = 0.5` el silencio que deja una nota tocada separada, si no pasa de la mitad del intervalo hasta el siguiente ataque, se escribe como parte de la nota. En esta grabación las corcheas se tocan separadas y la partitura las escribe como corcheas, así que la primera configuración las convierte en semicorchea más silencio.

La columna `coincidencia_alturas` compara solo la secuencia de notas sin su valor. Sirve para separar el error de altura, que viene del modelo acústico, del error rítmico, que viene del cuantizador.

In [3]:
resultados, por_categoria = [], []
for archivo, legato in [(a, l) for a in sorted(NOTAS.glob("*.json")) for l in (0.0, 0.5)]:
    datos = json.loads(archivo.read_text())
    pieza = piezas[datos["pieza"]]
    ev = evaluate_piece(pieza, datos["notas"], datos["duracion_s"], legato=legato)
    fila = {"modelo": datos["modelo"], "legato": legato, "nombre": nombres[pieza.index],
            "teclados": 2 if "2_Clav" in datos["archivo"] and "ovvero" not in datos["archivo"] else 1,
            "compas": f"{pieza.beats}/{pieza.beat_type}", "duracion_s": round(datos["duracion_s"], 1)}
    fila.update(ev.row())
    resultados.append(fila)
    for cat, s in ev.both.per_category.items():
        por_categoria.append({"modelo": datos["modelo"], "legato": legato, "pieza": pieza.index, "categoria": cat,
                              "ref": s.ref, "gen": s.hyp, "aciertos": s.matches,
                              "sustituciones": s.substitutions, "omisiones": s.deletions,
                              "inserciones": s.insertions})
    print(f"{datos['modelo']:9s} legato {legato}  {nombres[pieza.index]:14s} BSA {ev.both.bsa:6.2%}  alturas {ev.pitch_agreement:6.2%}")

df = pd.DataFrame(resultados)
cat = pd.DataFrame(por_categoria)
df.to_csv(SALIDA / "oe5_bsa_por_pieza.csv", index=False)
cat.to_csv(SALIDA / "oe5_bsa_por_categoria.csv", index=False)

ajustado  legato 0.0  Aria           BSA 25.44%  alturas 28.85%


ajustado  legato 0.5  Aria           BSA 25.82%  alturas 32.46%


ajustado  legato 0.0  Variación 1    BSA 20.93%  alturas 65.13%


ajustado  legato 0.5  Variación 1    BSA 31.82%  alturas 73.94%


ajustado  legato 0.0  Variación 2    BSA 21.12%  alturas 38.95%


...

original  legato 0.0  Variación 28   BSA 14.86%  alturas 32.01%


original  legato 0.5  Variación 28   BSA 15.35%  alturas 31.97%


original  legato 0.0  Variación 29   BSA 28.77%  alturas 17.02%


original  legato 0.5  Variación 29   BSA 27.11%  alturas 16.61%
original  legato 0.0  Variación 30   BSA 22.77%  alturas 38.38%


original  legato 0.5  Variación 30   BSA 22.99%  alturas 40.74%
original  legato 0.0  Aria da capo   BSA 20.30%  alturas 49.67%
original  legato 0.5  Aria da capo   BSA 22.80%  alturas 56.41%


## 4. Resultados por modelo

In [4]:
resumen = df.groupby(["modelo", "legato"]).agg(
    piezas=("pieza", "count"),
    bsa_medio=("bsa", "mean"), bsa_min=("bsa", "min"), bsa_max=("bsa", "max"),
    ponderado_medio=("bsa_ponderado", "mean"),
    bsa_md=("bsa_md", "mean"), bsa_mi=("bsa_mi", "mean"),
    alturas=("coincidencia_alturas", "mean"),
    piezas_sobre_70=("bsa", lambda s: int((s >= 0.70).sum())),
)
resumen.round(4)

piezas  bsa_medio  bsa_min  bsa_max  ponderado_medio  bsa_md  \
modelo   legato                                                                 
ajustado 0.0         31     0.2189   0.1478   0.3347           0.2326  0.2003   
         0.5         31     0.2390   0.1520   0.3580           0.2435  0.2072   
original 0.0         31     0.2166   0.1486   0.3446           0.2329  0.1951   
         0.5         31     0.2381   0.1535   0.3736           0.2426  0.2037   

                 bsa_mi  alturas  piezas_sobre_70  
modelo   legato                                    
ajustado 0.0     0.2191   0.3447                0  
         0.5     0.2567   0.3856                0  
original 0.0     0.2191   0.3378                0  
         0.5     0.2583   0.3758                0

In [5]:
tabla = df.pivot_table(index=["pieza", "nombre", "teclados"], columns=["modelo", "legato"], values="bsa").mul(100).round(2)
tabla

modelo                      ajustado        original       
legato                           0.0    0.5      0.0    0.5
pieza nombre       teclados                                
0     Aria         1           25.44  25.82    25.71  26.26
1     Variación 1  1           20.93  31.82    21.01  32.09
2     Variación 2  1           21.12  26.87    21.05  27.08
3     Variación 3  1           22.70  28.22    21.89  27.51
4     Variación 4  1           25.69  29.75    24.34  27.95
5     Variación 5  1           17.43  20.22    17.06  19.30
6     Variación 6  1           26.44  28.14    25.80  28.07
7     Variación 7  1           26.44  27.59    26.06  27.69
8     Variación 8  2           23.82  26.76    22.72  25.50
9     Variación 9  1           21.89  24.11    21.29  23.38
10    Variación 10 1           16.01  20.48    15.63  20.86
11    Variación 11 2           23.94  25.27    23.96  25.49
12    Variación 12 1           24.34  25.14    24.15  25.23
13    Variación 13 2           15.78  17.26    15.68  17.64
14    Variación 14 2           17.18  16.42    17.12  16.81
15    Variación 15 1           22.78  22.37    21.66  21.39
16    Variación 16 1           17.68  21.00    16.89  20.43
17    Variación 17 2           25.17  26.45    25.16  26.46
18    Variación 18 1           21.80  27.37    21.03  28.02
19    Variación 19 1           23.82  25.53    23.74  24.71
20    Variación 20 2           22.67  20.88    22.72  21.22
21    Variación 21 1           17.24  16.51    16.15  15.90
22    Variación 22 1           21.04  22.98    20.85  23.01
23    Variación 23 2           20.47  19.33    19.18  18.58
24    Variación 24 1           33.47  35.80    34.46  37.36
26    Variación 26 2           14.91  15.44    14.91  15.72
27    Variación 27 2           22.97  26.13    24.40  26.27
28    Variación 28 2           14.78  15.20    14.86  15.35
29    Variación 29 1           28.90  27.15    28.77  27.11
30    Variación 30 1           21.69  21.98    22.77  22.99
31    Aria da capo 1           19.98  22.91    20.30  22.80

## 5. Desviaciones por categoría

In [6]:
agregado = cat.groupby(["modelo", "legato", "categoria"])[["ref", "gen", "aciertos", "sustituciones", "omisiones", "inserciones"]].sum()
agregado["bsa"] = (agregado["aciertos"] / agregado[["ref", "gen"]].max(axis=1)).round(4)
agregado

ref    gen  aciertos  sustituciones  \
modelo   legato categoria                                             
ajustado 0.0    alteraciones   2221   3113       898           1064   
                bar_over_bar    120      3        11             62   
                barras           36      0         0             26   
                in_accords     2594      0         8           2024   
                intervalos      491   9275       244            175   
                ligaduras      2996   1824       605           1986   
                notas         37433  32736      7236          25337   
                octavas        8346   8974      3543           3824   
         0.5    alteraciones   2221   3143       897           1033   
                bar_over_bar    120      5        17             62   
                barras           36      0         0             27   
                in_accords     2594      0         8           1943   
                intervalos      491   9347       251            158   
                ligaduras      2996   2408       673           1894   
                notas         37433  30350      7910          24070   
                octavas        8346   9031      3605           3618   
original 0.0    alteraciones   2221   3142       883           1092   
                bar_over_bar    120      3        16             61   
                barras           36      0         0             24   
                in_accords     2594      0         7           2062   
                intervalos      491   9612       233            191   
                ligaduras      2996   1756       566           2034   
                notas         37433  32826      7207          25377   
                octavas        8346   9182      3574           3797   
         0.5    alteraciones   2221   3176       890           1042   
                bar_over_bar    120      4        16             63   
                barras           36      0         1             26   
                in_accords     2594      0         8           1954   
                intervalos      491   9699       240            172   
                ligaduras      2996   2316       665           1911   
                notas         37433  30365      7951          24020   
                octavas        8346   9244      3626           3580   

                              omisiones  inserciones     bsa  
modelo   legato categoria                                     
ajustado 0.0    alteraciones        259          505  0.2885  
                bar_over_bar         47            1  0.0917  
                barras               10            0  0.0000  
                in_accords          562            0  0.0031  
                intervalos           72         2335  0.0263  
                ligaduras           405          235  0.2019  
                notas              4860         4491  0.1933  
                octavas             979         1315  0.3948  
         0.5    alteraciones        291          472  0.2854  
                bar_over_bar         41            1  0.1417  
                barras                9            0  0.0000  
                in_accords          643            0  0.0031  
                intervalos           82         2306  0.0269  
                ligaduras           429          304  0.2246  
                notas              5453         3755  0.2113  
                octavas            1123         1280  0.3992  
original 0.0    alteraciones        246          524  0.2810  
                bar_over_bar         43            0  0.1333  
                barras               12            0  0.0000  
                in_accords          525            0  0.0027  
                intervalos           67         2520  0.0242  
                ligaduras           396          241  0.1889  
                notas              4849         4660  0.1925  
                octavas             975   

## 6. Uno o dos teclados

In [7]:
df.groupby(["modelo", "legato", "teclados"])["bsa"].agg(["count", "mean"]).round(4)

count    mean
modelo   legato teclados               
ajustado 0.0    1            21  0.2271
                2            10  0.2017
         0.5    1            21  0.2532
                2            10  0.2091
original 0.0    1            21  0.2241
                2            10  0.2007
         0.5    1            21  0.2520
                2            10  0.2090

### Observaciones

*(Completar al correr.)*